[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philmui/ai-foundations/blob/main/01_architecture_of_data/tutorial.ipynb)

## ▶️ Run this notebook in Google Colab

**Option A — One click (recommended):** Click the **Open in Colab** badge above. It opens this notebook directly in a free cloud runtime — nothing to install locally.

**Option B — Upload manually:**
1. Go to [colab.research.google.com](https://colab.research.google.com).
2. Choose **File → Upload notebook** and select this `tutorial.ipynb`.

**Then, in Colab:**
1. Run the **first code cell** (the dependency-setup cell) — it installs everything this notebook needs directly into the Colab kernel. No `pip install`, `uv sync`, or `pyproject.toml` required.
2. Run the remaining cells top to bottom via **Runtime → Run all** (or `Ctrl/Cmd+F9`).

> 💡 This notebook is **fully self-contained**: any data files it uses are generated inside the notebook, so it runs end-to-end in Colab with no external files.

---

In [ ]:
# === Inline dependency setup (self-contained; no `uv sync` / pyproject.toml needed) ===
# Installs this notebook's libraries directly into the running kernel.
# Works in local Jupyter, VS Code, and Google Colab. Safe to re-run (idempotent).
import sys, subprocess

_DEPS = [
    'python-dotenv>=1.0.0',
    'numpy>=1.26.0',
    'matplotlib>=3.7',
]

# Ensure pip is available in this kernel (some minimal venvs ship without it), then install.
try:
    import pip  # noqa: F401
except ModuleNotFoundError:
    import ensurepip; ensurepip.bootstrap()

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_DEPS])
print('\u2713 Dependencies installed:', ', '.join(_DEPS))


In [ ]:
# === Google Colab: native Mermaid diagram rendering ===
# Colab does not render ```mermaid Markdown blocks. This helper injects the
# official Mermaid.js library into a cell's output and draws the diagram there.
# The Markdown cells keep their ```mermaid blocks so GitHub still renders them;
# the code cells below each diagram call render_mermaid(...) so Colab does too.
# Safe to re-run.
import IPython.display as _ipd

_MERMAID_COUNTER = 0


def render_mermaid(graph: str):
    # Render a Mermaid diagram in the current cell's output (works in Colab).
    global _MERMAID_COUNTER
    _MERMAID_COUNTER += 1
    container_id = f"mermaid-diagram-{_MERMAID_COUNTER}"
    # The graph text is placed verbatim inside the .mermaid div; Mermaid reads it
    # as textContent, so it must NOT be HTML-escaped.
    html = f'''
<div id="{container_id}" class="mermaid">
{graph}
</div>
<script type="module">
  import mermaid from "https://cdn.jsdelivr.net/npm/mermaid@11/dist/mermaid.esm.min.mjs";
  mermaid.initialize({{ startOnLoad: false }});
  const el = document.getElementById("{container_id}");
  await mermaid.run({{ nodes: [el] }});
</script>
'''
    _ipd.display(_ipd.HTML(html))


# Module 1: The Architecture of Data
## Lists, Dictionaries, and Computational Complexity

**Learning Objectives:**
- Understand how Python lists and dictionaries work under the hood
- Distinguish between mutable and immutable objects
- Apply Big O notation to analyze data structure performance
- Build a practical tokenizer dictionary for NLP/AI applications

---

## 1. Introduction: Why Data Structures Matter for AI

Before you can build transformers, fine-tune Large Language Models (LLMs), or train neural networks, you must understand **how data is stored and accessed efficiently**.

### The Real-World Problem

Imagine you're working with GPT-4, which has a vocabulary of **100,000+ tokens**. Every time the model processes a word, it needs to:
1. Look up the word in its vocabulary
2. Find the corresponding token ID
3. Retrieve the embedding vector for that token

If each lookup takes even 0.001 seconds using a poor data structure, processing a 1000-word document would take **1+ seconds** just for vocabulary lookups. With the right data structure, this drops to **microseconds**.

> **Key Insight:** The choice between a list and a dictionary for vocabulary lookup can mean the difference between a model that processes text in real-time and one that's unusably slow. This is why data structures are foundational to AI engineering.

In this module, we'll build the intuition and practical skills to make these critical engineering decisions.

In [ ]:
# Setup: Import all required libraries
from dotenv import load_dotenv, find_dotenv
import time
import random
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

# Load environment variables
load_dotenv(find_dotenv())

# Configure matplotlib for clean plots
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print("✓ Environment configured successfully")

---
## 2. Python Lists Deep Dive

### What is a List?

A Python list is an **ordered, mutable sequence** of elements. Under the hood, it's implemented as a **dynamic array** that stores references to objects.

### Key Properties:
- **Ordered:** Elements maintain their insertion order
- **Mutable:** You can change, add, or remove elements after creation
- **Heterogeneous:** Can contain different data types
- **Indexed:** Access elements by position (0-indexed)

### Memory Model

```
List:    [  ref1  |  ref2  |  ref3  |  ref4  ]
            ↓         ↓         ↓         ↓
Memory:  ["hello"] [42]    [3.14]   [True]
```

The list itself is a contiguous block of memory addresses (references), but the actual objects can be scattered throughout memory.

In [ ]:
# Creating and manipulating lists
vocabulary = ["hello", "world", "artificial", "intelligence", "neural", "network"]

# Indexing (O(1) access)
print(f"First word: {vocabulary[0]}")
print(f"Last word: {vocabulary[-1]}")

# Slicing (creates a new list)
print(f"\nFirst three words: {vocabulary[:3]}")
print(f"Every other word: {vocabulary[::2]}")

# Mutation (in-place modification)
vocabulary.append("transformer")
print(f"\nAfter append: {vocabulary}")

# List comprehension (Pythonic way to create lists)
word_lengths = [len(word) for word in vocabulary]
print(f"\nWord lengths: {word_lengths}")

### The Performance Problem with Lists

When you search for an element in a list using `x in my_list`, Python must check **every element** sequentially until it finds a match (or reaches the end). This is called **linear search** and has **O(n) time complexity**.

> **Key Insight:** For a vocabulary of 100,000 words, the average lookup requires checking 50,000 elements. This becomes a bottleneck in production AI systems.

Let's measure this:

In [ ]:
# Create a large vocabulary list (simulating a real tokenizer vocabulary)
large_vocab_list = [f"token_{i}" for i in range(100000)]

# Search for a token near the end (worst case)
target_token = "token_99999"

# Time the lookup
start = time.perf_counter()
result = target_token in large_vocab_list
end = time.perf_counter()

list_lookup_time = (end - start) * 1000  # Convert to milliseconds
print(f"List lookup time: {list_lookup_time:.4f} ms")
print(f"Result: {result}")

---
## 3. Python Dictionaries Deep Dive

### What is a Dictionary?

A Python dictionary is an **unordered collection of key-value pairs** implemented using a **hash table**. This gives it exceptional lookup performance.

### How Hash Tables Work

```mermaid
graph LR
    A["Key: 'hello'"] --> B[Hash Function]
    B --> C["Hash: 5238423984"]
    C --> D["Index: 423984 % 8 = 0"]
    D --> E["Bucket 0: ('hello', 1045)"]
    
    style A fill:#e1f5ff
    style B fill:#fff4e1
    style C fill:#ffe1f5
    style D fill:#e1ffe1
    style E fill:#f5e1ff
```

### Key Properties:
- **O(1) average lookup time:** Constant time, regardless of dictionary size
- **Mutable:** Can add, modify, or remove key-value pairs
- **Keys must be immutable:** Strings, numbers, tuples (not lists)
- **Keys are unique:** Each key appears only once

> **Key Insight:** A hash function converts a key into an integer, which is used to calculate an array index. This allows direct access to the value without searching through all entries.

In [ ]:
# Colab-native rendering of the Mermaid diagram(s) in the cell above.
# (The Markdown block still renders natively on GitHub.)
render_mermaid(r'''
graph LR
    A["Key: 'hello'"] --> B[Hash Function]
    B --> C["Hash: 5238423984"]
    C --> D["Index: 423984 % 8 = 0"]
    D --> E["Bucket 0: ('hello', 1045)"]
    
    style A fill:#e1f5ff
    style B fill:#fff4e1
    style C fill:#ffe1f5
    style D fill:#e1ffe1
    style E fill:#f5e1ff
''')


In [ ]:
# Creating a tokenizer dictionary (word -> token ID)
token_dict = {
    "hello": 1045,
    "world": 2087,
    "artificial": 5423,
    "intelligence": 9234,
    "neural": 3421,
    "network": 7834
}

# O(1) lookup
print(f"Token ID for 'neural': {token_dict['neural']}")

# Safe lookup with .get() (returns None if key doesn't exist)
print(f"Token ID for 'transformer': {token_dict.get('transformer', 'NOT_FOUND')}")

# Adding new entries
token_dict["transformer"] = 12456
print(f"\nUpdated dictionary: {token_dict}")

# Iterating over dictionaries
print("\nAll tokens:")
for word, token_id in token_dict.items():
    print(f"  {word:15s} -> {token_id}")

### Dictionary vs List: The Performance Comparison

Let's compare lookup times for a large vocabulary (100,000 tokens), which is realistic for modern NLP systems.

> **Key Insight:** When working with millions of tokens (like processing a large dataset), the difference between O(n) and O(1) lookup is the difference between hours and seconds.

In [ ]:
# Create both data structures with the same data
vocab_size = 100000
large_vocab_list = [f"token_{i}" for i in range(vocab_size)]
large_vocab_dict = {f"token_{i}": i for i in range(vocab_size)}

# Test lookups at different positions
test_positions = [100, 1000, 10000, 50000, 99999]
list_times = []
dict_times = []

print("Comparing lookup times (in milliseconds):\n")
print(f"{'Position':<10} {'List Lookup':<15} {'Dict Lookup':<15} {'Speedup':<10}")
print("-" * 55)

for pos in test_positions:
    target = f"token_{pos}"
    
    # Time list lookup
    start = time.perf_counter()
    _ = target in large_vocab_list
    list_time = (time.perf_counter() - start) * 1000
    list_times.append(list_time)
    
    # Time dict lookup
    start = time.perf_counter()
    _ = target in large_vocab_dict
    dict_time = (time.perf_counter() - start) * 1000
    dict_times.append(dict_time)
    
    speedup = list_time / dict_time if dict_time > 0 else float('inf')
    print(f"{pos:<10} {list_time:<15.6f} {dict_time:<15.6f} {speedup:<10.1f}x")

In [ ]:
# Visualize the performance difference
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart comparing lookup times
x = np.arange(len(test_positions))
width = 0.35

ax1.bar(x - width/2, list_times, width, label='List Lookup', color='#ff6b6b', alpha=0.8)
ax1.bar(x + width/2, dict_times, width, label='Dict Lookup', color='#4ecdc4', alpha=0.8)

ax1.set_xlabel('Token Position in Vocabulary', fontweight='bold')
ax1.set_ylabel('Lookup Time (ms)', fontweight='bold')
ax1.set_title('List vs Dictionary Lookup Performance', fontweight='bold', fontsize=13)
ax1.set_xticks(x)
ax1.set_xticklabels([f"{p:,}" for p in test_positions])
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Speedup visualization
speedups = [lt / dt if dt > 0 else 0 for lt, dt in zip(list_times, dict_times)]
ax2.plot(test_positions, speedups, marker='o', linewidth=2, markersize=8, color='#9b59b6')
ax2.fill_between(test_positions, speedups, alpha=0.3, color='#9b59b6')
ax2.set_xlabel('Token Position in Vocabulary', fontweight='bold')
ax2.set_ylabel('Speedup Factor (x)', fontweight='bold')
ax2.set_title('Dictionary Speedup Over List', fontweight='bold', fontsize=13)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n✓ Dictionary lookups are consistently ~{np.mean(speedups):.0f}x faster!")

---
## 4. Mutable vs Immutable Objects

Understanding mutability is critical for debugging AI code, especially when working with PyTorch tensors or when building data pipelines.

### What is Mutability?

- **Mutable objects** can be changed after creation (their content can be modified in-place)
- **Immutable objects** cannot be changed after creation (any "modification" creates a new object)

### Python's Mutable vs Immutable Types

| Mutable | Immutable |
|---------|----------|
| `list` | `int` |
| `dict` | `float` |
| `set` | `str` |
| NumPy arrays | `tuple` |
| PyTorch tensors | `frozenset` |

> **Key Insight:** Understanding mutability prevents bugs where you accidentally modify shared data. In ML pipelines, this is crucial when batching data or applying transformations.

In [ ]:
# Demonstrating mutability with id()
# id() returns the memory address of an object

print("=== MUTABLE: List ===")
vocab = ["hello", "world"]
print(f"Original list: {vocab}")
print(f"Memory address: {id(vocab)}")

# In-place modification (same object)
vocab.append("AI")
print(f"\nAfter append: {vocab}")
print(f"Memory address: {id(vocab)}")
print("✓ Same memory address = mutation happened in-place\n")

print("=== IMMUTABLE: String ===")
text = "hello"
print(f"Original string: {text}")
print(f"Memory address: {id(text)}")

# "Modification" creates a new object
text = text + " world"
print(f"\nAfter concatenation: {text}")
print(f"Memory address: {id(text)}")
print("✓ Different memory address = new object created")

### The Aliasing Problem

When you assign a mutable object to a new variable, both variables reference the **same object in memory**. This is called **aliasing** and is a common source of bugs.

In [ ]:
# The aliasing trap with mutable objects
original_vocab = ["neural", "network"]
alias_vocab = original_vocab  # Both variables point to the same list!

print("Before modification:")
print(f"original_vocab: {original_vocab}")
print(f"alias_vocab:    {alias_vocab}")
print(f"Same object? {id(original_vocab) == id(alias_vocab)}")

# Modify through the alias
alias_vocab.append("transformer")

print("\nAfter modifying alias_vocab:")
print(f"original_vocab: {original_vocab}")
print(f"alias_vocab:    {alias_vocab}")
print("⚠️  Both changed! They're the same object.")

# Solution: Create a true copy
print("\n=== Creating a True Copy ===")
original_vocab = ["neural", "network"]
true_copy = original_vocab.copy()  # or list(original_vocab) or original_vocab[:]

true_copy.append("transformer")
print(f"original_vocab: {original_vocab}")
print(f"true_copy:      {true_copy}")
print(f"Same object? {id(original_vocab) == id(true_copy)}")
print("✓ Only the copy changed!")

### Why This Matters for AI/ML

**Common bug in data preprocessing:**
```python
def preprocess_batch(data):
    data.append(pad_token)  # Mutates the original data!
    return data

train_data = [1, 2, 3]
batch1 = preprocess_batch(train_data)  # train_data is now [1, 2, 3, 0]
batch2 = preprocess_batch(train_data)  # train_data is now [1, 2, 3, 0, 0]
# Each call adds another pad token to the original data!
```

**Correct approach:**
```python
def preprocess_batch(data):
    data_copy = data.copy()  # Create a copy first
    data_copy.append(pad_token)
    return data_copy
```

> **Key Insight:** Always ask: "Am I modifying the original data or creating a new copy?" This is especially critical when building PyTorch DataLoaders or preprocessing pipelines.

---
## 5. Nested Dictionaries and JSON

Real-world AI systems work with structured data, often in JSON format. Understanding nested dictionaries is essential for working with API responses, configuration files, and dataset metadata.

### Example: OpenAI API Response Structure

When you call an LLM API, you get back nested JSON data:

In [ ]:
# Simulated OpenAI API response (simplified)
api_response = {
    "id": "chatcmpl-123",
    "object": "chat.completion",
    "created": 1677652288,
    "model": "gpt-4",
    "usage": {
        "prompt_tokens": 56,
        "completion_tokens": 31,
        "total_tokens": 87
    },
    "choices": [
        {
            "message": {
                "role": "assistant",
                "content": "Data structures are fundamental to AI."
            },
            "finish_reason": "stop",
            "index": 0
        }
    ]
}

# Accessing nested values
print("=== Extracting Information from Nested Dict ===")
print(f"Model used: {api_response['model']}")
print(f"Total tokens: {api_response['usage']['total_tokens']}")
print(f"Response: {api_response['choices'][0]['message']['content']}")

# Safe access with .get() to avoid KeyError
print(f"\nFinish reason: {api_response['choices'][0].get('finish_reason', 'unknown')}")
print(f"Temperature: {api_response.get('temperature', 'not specified')}")

### Building a Tokenizer Metadata Dictionary

Let's create a more complex nested structure to represent tokenizer metadata:

In [ ]:
# Tokenizer configuration with nested dictionaries
tokenizer_config = {
    "metadata": {
        "name": "custom_tokenizer_v1",
        "vocab_size": 50000,
        "created": "2026-06-21",
        "special_tokens": {
            "pad_token": "[PAD]",
            "unk_token": "[UNK]",
            "bos_token": "[BOS]",
            "eos_token": "[EOS]"
        }
    },
    "vocab": {
        "[PAD]": 0,
        "[UNK]": 1,
        "[BOS]": 2,
        "[EOS]": 3,
        "the": 4,
        "neural": 5,
        "network": 6
    },
    "statistics": {
        "most_frequent": [("the", 15234), ("neural", 8456), ("network", 7821)],
        "avg_token_length": 5.2
    }
}

# Pretty print the structure
import json
print("Tokenizer Configuration:")
print(json.dumps(tokenizer_config, indent=2))

In [ ]:
# Accessing and manipulating nested data
print("=== Working with Nested Data ===")

# Extract special tokens
special_tokens = tokenizer_config["metadata"]["special_tokens"]
print(f"Special tokens: {list(special_tokens.keys())}")

# Get token ID (with fallback to UNK)
unk_id = tokenizer_config["vocab"]["[UNK]"]

def get_token_id(word):
    return tokenizer_config["vocab"].get(word, unk_id)

print(f"\nToken ID for 'neural': {get_token_id('neural')}")
print(f"Token ID for 'transformer': {get_token_id('transformer')} (unknown word)")

# Update statistics
tokenizer_config["statistics"]["total_documents_processed"] = 10000
print(f"\nUpdated statistics: {tokenizer_config['statistics']}")

---
## 6. Big O Notation for AI

Big O notation describes how the runtime of an algorithm grows as the input size increases. For AI systems processing millions of data points, this matters **enormously**.

### Common Time Complexities

```mermaid
graph TD
    A["Algorithm Complexity"] --> B["O(1) - Constant"]
    A --> C["O(log n) - Logarithmic"]
    A --> D["O(n) - Linear"]
    A --> E["O(n log n) - Linearithmic"]
    A --> F["O(n²) - Quadratic"]
    A --> G["O(2ⁿ) - Exponential"]
    
    B --> B1["Dict lookup<br/>Array index access"]
    C --> C1["Binary search<br/>Tree operations"]
    D --> D1["List search<br/>Single loop"]
    E --> E1["Efficient sorting<br/>Merge sort"]
    F --> F1["Nested loops<br/>Bubble sort"]
    G --> G1["Recursive branching<br/>Brute force"]
    
    style B fill:#4ecdc4
    style C fill:#95e1d3
    style D fill:#f38181
    style E fill:#ffa07a
    style F fill:#ff6b6b
    style G fill:#aa4545
```

### Real-World Example: Token Lookup

| Data Structure | Operation | Time Complexity | 1M Lookups |
|---------------|-----------|----------------|------------|
| Dictionary | `word in vocab_dict` | O(1) | ~0.01s |
| List | `word in vocab_list` | O(n) | ~5-10s |
| Sorted List + Binary Search | `bisect.bisect(list, word)` | O(log n) | ~0.2s |

> **Key Insight:** For LLM inference processing 1000 tokens/second, the difference between O(1) and O(n) lookup is the difference between real-time processing and 5-10x slowdown.

In [ ]:
# Colab-native rendering of the Mermaid diagram(s) in the cell above.
# (The Markdown block still renders natively on GitHub.)
render_mermaid(r'''
graph TD
    A["Algorithm Complexity"] --> B["O(1) - Constant"]
    A --> C["O(log n) - Logarithmic"]
    A --> D["O(n) - Linear"]
    A --> E["O(n log n) - Linearithmic"]
    A --> F["O(n²) - Quadratic"]
    A --> G["O(2ⁿ) - Exponential"]
    
    B --> B1["Dict lookup<br/>Array index access"]
    C --> C1["Binary search<br/>Tree operations"]
    D --> D1["List search<br/>Single loop"]
    E --> E1["Efficient sorting<br/>Merge sort"]
    F --> F1["Nested loops<br/>Bubble sort"]
    G --> G1["Recursive branching<br/>Brute force"]
    
    style B fill:#4ecdc4
    style C fill:#95e1d3
    style D fill:#f38181
    style E fill:#ffa07a
    style F fill:#ff6b6b
    style G fill:#aa4545
''')


In [ ]:
# Visualizing Big O growth
n_values = np.array([10, 50, 100, 500, 1000, 5000, 10000])

# Define complexity functions
o_1 = np.ones_like(n_values)  # Constant
o_log_n = np.log2(n_values)   # Logarithmic
o_n = n_values                # Linear
o_n_log_n = n_values * np.log2(n_values)  # Linearithmic
o_n2 = n_values ** 2          # Quadratic

# Plot
plt.figure(figsize=(12, 7))
plt.plot(n_values, o_1, 'o-', label='O(1) - Dict lookup', linewidth=2, markersize=6, color='#4ecdc4')
plt.plot(n_values, o_log_n, 's-', label='O(log n) - Binary search', linewidth=2, markersize=6, color='#95e1d3')
plt.plot(n_values, o_n, '^-', label='O(n) - List search', linewidth=2, markersize=6, color='#ffa07a')
plt.plot(n_values, o_n_log_n, 'D-', label='O(n log n) - Merge sort', linewidth=2, markersize=6, color='#ff6b6b')
plt.plot(n_values, o_n2, 'v-', label='O(n²) - Nested loops', linewidth=2, markersize=6, color='#aa4545')

plt.xlabel('Input Size (n)', fontsize=12, fontweight='bold')
plt.ylabel('Operations', fontsize=12, fontweight='bold')
plt.title('Big O Complexity Growth Comparison', fontsize=14, fontweight='bold')
plt.legend(loc='upper left', fontsize=10)
plt.grid(True, alpha=0.3)
plt.yscale('log')  # Logarithmic scale to show all curves clearly
plt.tight_layout()
plt.show()

print("Notice how O(1) stays flat while O(n²) explodes!")

### Practical Timing Experiment

Let's measure actual execution times for different complexities:

In [ ]:
# O(1): Dictionary lookup
def dict_lookup_experiment(n):
    vocab = {f"word_{i}": i for i in range(n)}
    target = f"word_{n-1}"  # Worst case
    
    start = time.perf_counter()
    _ = target in vocab
    return time.perf_counter() - start

# O(n): List search
def list_search_experiment(n):
    vocab = [f"word_{i}" for i in range(n)]
    target = f"word_{n-1}"  # Worst case
    
    start = time.perf_counter()
    _ = target in vocab
    return time.perf_counter() - start

# O(n²): Nested loop (find all pairs)
def nested_loop_experiment(n):
    data = list(range(n))
    
    start = time.perf_counter()
    count = 0
    for i in data:
        for j in data:
            count += 1
    return time.perf_counter() - start

# Run experiments
sizes = [100, 500, 1000, 2000, 5000]
dict_times = [dict_lookup_experiment(n) * 1000 for n in sizes]  # Convert to ms
list_times = [list_search_experiment(n) * 1000 for n in sizes]
nested_times = [nested_loop_experiment(n) * 1000 for n in sizes]

# Plot results
plt.figure(figsize=(12, 6))
plt.plot(sizes, dict_times, 'o-', label='O(1) - Dict', linewidth=2, markersize=8, color='#4ecdc4')
plt.plot(sizes, list_times, 's-', label='O(n) - List', linewidth=2, markersize=8, color='#ffa07a')
plt.plot(sizes, nested_times, '^-', label='O(n²) - Nested loop', linewidth=2, markersize=8, color='#ff6b6b')

plt.xlabel('Vocabulary Size', fontsize=12, fontweight='bold')
plt.ylabel('Time (milliseconds)', fontsize=12, fontweight='bold')
plt.title('Actual Runtime Comparison: O(1) vs O(n) vs O(n²)', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.tight_layout()
plt.show()

print("\nAt n=5000:")
print(f"  Dict (O(1)):        {dict_times[-1]:.6f} ms")
print(f"  List (O(n)):        {list_times[-1]:.4f} ms")
print(f"  Nested loop (O(n²)): {nested_times[-1]:.2f} ms")
print(f"\n  List is {list_times[-1]/dict_times[-1]:.0f}x slower than dict")
print(f"  Nested loop is {nested_times[-1]/list_times[-1]:.0f}x slower than list")

---
## 7. Lab: Building a Tokenizer Dictionary

Now let's apply everything we've learned to build a practical tokenizer dictionary from scratch. This is the foundation of every NLP system.

### The Pipeline

```mermaid
graph LR
    A["Raw Text<br/>'Neural networks are powerful'"] --> B["Split into Words<br/>['neural', 'networks', 'are', 'powerful']"]
    B --> C["Count Frequencies<br/>{'neural': 2, 'networks': 3, ...}"]
    C --> D["Assign Token IDs<br/>{'neural': 0, 'networks': 1, ...}"]
    D --> E["Token Dictionary<br/>O(1) lookup"]
    
    style A fill:#e1f5ff
    style B fill:#fff4e1
    style C fill:#ffe1f5
    style D fill:#e1ffe1
    style E fill:#f5e1ff
```

### Sample Corpus

We'll use a small AI-themed corpus to demonstrate the concepts:

In [ ]:
# Colab-native rendering of the Mermaid diagram(s) in the cell above.
# (The Markdown block still renders natively on GitHub.)
render_mermaid(r'''
graph LR
    A["Raw Text<br/>'Neural networks are powerful'"] --> B["Split into Words<br/>['neural', 'networks', 'are', 'powerful']"]
    B --> C["Count Frequencies<br/>{'neural': 2, 'networks': 3, ...}"]
    C --> D["Assign Token IDs<br/>{'neural': 0, 'networks': 1, ...}"]
    D --> E["Token Dictionary<br/>O(1) lookup"]
    
    style A fill:#e1f5ff
    style B fill:#fff4e1
    style C fill:#ffe1f5
    style D fill:#e1ffe1
    style E fill:#f5e1ff
''')


In [ ]:
# Sample text corpus about AI and neural networks
corpus = """
Neural networks are computational models inspired by biological neural networks.
Deep learning uses neural networks with multiple layers to learn hierarchical representations.
Transformers are neural network architectures that use attention mechanisms.
The attention mechanism allows models to focus on relevant parts of the input.
Large language models like GPT are based on transformer architectures.
Training neural networks requires large datasets and computational resources.
Backpropagation is the algorithm used to train neural networks efficiently.
Neural networks can learn complex patterns from data through iterative training.
Modern AI systems rely heavily on neural network architectures and deep learning.
The development of neural networks has revolutionized artificial intelligence.
"""

print("Sample corpus loaded:")
print(f"  Length: {len(corpus)} characters")
print(f"  First 150 chars: {corpus[:150]}...")

### Step 1: Text Preprocessing and Word Extraction

In [ ]:
import re

# Preprocessing: lowercase and split into words
def tokenize_text(text):
    """
    Convert text to lowercase and split into words.
    Remove punctuation and extra whitespace.
    """
    # Convert to lowercase
    text = text.lower()
    # Remove punctuation and split into words
    words = re.findall(r'\b[a-z]+\b', text)
    return words

# Extract all words from the corpus
all_words = tokenize_text(corpus)

print(f"Total words extracted: {len(all_words)}")
print(f"\nFirst 20 words: {all_words[:20]}")
print(f"\nUnique words: {len(set(all_words))}")

### Step 2: Build Frequency Counter

In [ ]:
# Count word frequencies
word_freq = Counter(all_words)

print("Word Frequency Distribution:")
print(f"Total unique words: {len(word_freq)}\n")

# Show top 15 most frequent words
print("Top 15 most frequent words:")
for word, count in word_freq.most_common(15):
    print(f"  {word:15s} {count:3d} {'█' * count}")

# Visualize frequency distribution
top_words = word_freq.most_common(20)
words, freqs = zip(*top_words)

plt.figure(figsize=(12, 6))
plt.bar(words, freqs, color='#4ecdc4', alpha=0.8, edgecolor='black', linewidth=1.2)
plt.xlabel('Words', fontsize=12, fontweight='bold')
plt.ylabel('Frequency', fontsize=12, fontweight='bold')
plt.title('Top 20 Most Frequent Words in Corpus', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### Step 3: Assign Token IDs (Building the Dictionary)

In [ ]:
# Build the tokenizer dictionary
# Strategy: Reserve IDs 0-3 for special tokens, then assign by frequency

# Special tokens (standard in most tokenizers)
SPECIAL_TOKENS = {
    "[PAD]": 0,  # Padding token (for batching sequences of different lengths)
    "[UNK]": 1,  # Unknown token (for words not in vocabulary)
    "[BOS]": 2,  # Beginning of sequence
    "[EOS]": 3   # End of sequence
}

# Build word -> token_id mapping
word_to_id = SPECIAL_TOKENS.copy()
next_id = len(SPECIAL_TOKENS)

# Assign IDs to words by frequency (most frequent get lower IDs)
for word, freq in word_freq.most_common():
    if word not in word_to_id:
        word_to_id[word] = next_id
        next_id += 1

# Build reverse mapping: token_id -> word
id_to_word = {token_id: word for word, token_id in word_to_id.items()}

print(f"Tokenizer Dictionary Built!")
print(f"  Vocabulary size: {len(word_to_id)}")
print(f"  Special tokens: {len(SPECIAL_TOKENS)}")
print(f"  Regular words: {len(word_to_id) - len(SPECIAL_TOKENS)}")

print("\nSpecial tokens:")
for token, id_ in SPECIAL_TOKENS.items():
    print(f"  {token:10s} -> {id_}")

print("\nSample word mappings (most frequent):")
for word, freq in word_freq.most_common(10):
    print(f"  {word:15s} -> {word_to_id[word]:3d} (frequency: {freq})")

### Step 4: Tokenizer Class with Encode/Decode Functions

In [ ]:
class SimpleTokenizer:
    """A simple word-level tokenizer for demonstration purposes."""
    
    def __init__(self, word_to_id, id_to_word):
        self.word_to_id = word_to_id
        self.id_to_word = id_to_word
        self.vocab_size = len(word_to_id)
        self.unk_id = word_to_id["[UNK]"]
        self.pad_id = word_to_id["[PAD]"]
    
    def encode(self, text):
        """Convert text to a list of token IDs."""
        words = tokenize_text(text)
        return [self.word_to_id.get(word, self.unk_id) for word in words]
    
    def decode(self, token_ids):
        """Convert a list of token IDs back to text."""
        words = [self.id_to_word.get(id_, "[UNK]") for id_ in token_ids]
        return " ".join(words)
    
    def __len__(self):
        return self.vocab_size

# Create tokenizer instance
tokenizer = SimpleTokenizer(word_to_id, id_to_word)

print(f"Tokenizer created with vocabulary size: {len(tokenizer)}")
print("\n=== Testing Encode/Decode ===")

# Test encoding
test_sentence = "Neural networks learn from data using backpropagation."
encoded = tokenizer.encode(test_sentence)

print(f"\nOriginal: {test_sentence}")
print(f"Encoded:  {encoded}")
print(f"Decoded:  {tokenizer.decode(encoded)}")

# Test with unknown words
test_with_unknown = "Quantum computers use qubits for superposition."
encoded_unknown = tokenizer.encode(test_with_unknown)

print(f"\nWith unknown words: {test_with_unknown}")
print(f"Encoded: {encoded_unknown}")
print(f"Decoded: {tokenizer.decode(encoded_unknown)}")
print("  (Notice [UNK] tokens for words not in vocabulary)")

### Step 5: Demonstrate Lookup Speed

Now let's demonstrate why using a dictionary is crucial for tokenization performance:

In [ ]:
# Simulate large-scale tokenization
# Generate a long document with random words from our vocabulary
vocab_words = [w for w in word_to_id.keys() if not w.startswith('[')]
large_document = " ".join(random.choices(vocab_words, k=10000))  # 10,000 words

print(f"Generated test document with {len(large_document.split())} words")
print(f"Document preview: {large_document[:100]}...\n")

# Method 1: Using dictionary (O(1) lookup) - CORRECT
def tokenize_with_dict(text, tokenizer):
    return tokenizer.encode(text)

# Method 2: Using list (O(n) lookup) - INEFFICIENT
def tokenize_with_list(text, vocab_list, unk_id):
    words = tokenize_text(text)
    token_ids = []
    for word in words:
        try:
            idx = vocab_list.index(word)  # O(n) operation!
            token_ids.append(idx)
        except ValueError:
            token_ids.append(unk_id)
    return token_ids

vocab_list = list(word_to_id.keys())

# Benchmark
print("=== Performance Comparison ===")
print(f"Document size: {len(large_document.split())} words\n")

# Dictionary-based tokenization
start = time.perf_counter()
tokens_dict = tokenize_with_dict(large_document, tokenizer)
dict_time = time.perf_counter() - start

print(f"Dictionary tokenization (O(1) lookup): {dict_time*1000:.2f} ms")

# List-based tokenization
start = time.perf_counter()
tokens_list = tokenize_with_list(large_document, vocab_list, tokenizer.unk_id)
list_time = time.perf_counter() - start

print(f"List tokenization (O(n) lookup):      {list_time*1000:.2f} ms")
print(f"\nSpeedup: {list_time/dict_time:.1f}x faster with dictionary!")
print(f"\nFor a production LLM processing millions of tokens,")
print(f"this difference scales to hours vs. minutes of processing time.")

### Tokenizer Statistics and Analysis

In [ ]:
# Analyze tokenizer properties
print("=== Tokenizer Statistics ===")
print(f"\nVocabulary composition:")
print(f"  Total vocabulary size: {len(tokenizer)}")
print(f"  Special tokens:        {len(SPECIAL_TOKENS)}")
print(f"  Regular words:         {len(tokenizer) - len(SPECIAL_TOKENS)}")

# Word length distribution
word_lengths = [len(word) for word in vocab_words]
print(f"\nWord length statistics:")
print(f"  Average length: {np.mean(word_lengths):.2f} characters")
print(f"  Min length:     {min(word_lengths)} characters")
print(f"  Max length:     {max(word_lengths)} characters")

# Visualize word length distribution
plt.figure(figsize=(10, 5))
plt.hist(word_lengths, bins=range(1, max(word_lengths)+2), 
         color='#4ecdc4', alpha=0.7, edgecolor='black')
plt.xlabel('Word Length (characters)', fontsize=12, fontweight='bold')
plt.ylabel('Frequency', fontsize=12, fontweight='bold')
plt.title('Distribution of Word Lengths in Vocabulary', fontsize=13, fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Token ID distribution visualization
sample_text = " ".join(all_words[:100])  # First 100 words from corpus
sample_tokens = tokenizer.encode(sample_text)

plt.figure(figsize=(12, 4))
plt.plot(sample_tokens, 'o-', color='#4ecdc4', markersize=4, linewidth=1)
plt.xlabel('Position in Text', fontsize=12, fontweight='bold')
plt.ylabel('Token ID', fontsize=12, fontweight='bold')
plt.title('Token ID Sequence (First 100 Words)', fontsize=13, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n✓ Tokenizer analysis complete!")

---
## Summary & Key Takeaways

### Core Concepts Mastered

1. **Lists vs Dictionaries**
   - Lists: O(n) search, ordered, sequential access
   - Dictionaries: O(1) lookup, hash-based, key-value pairs
   - **Use dictionaries for lookups, lists for ordered sequences**

2. **Mutability Matters**
   - Mutable objects (list, dict) can be modified in-place
   - Immutable objects (str, tuple, int) create new objects on "modification"
   - **Always ask: am I mutating or copying?**

3. **Big O Notation**
   - O(1): Constant time (dict lookup, array indexing)
   - O(n): Linear time (list search, single loop)
   - O(n²): Quadratic time (nested loops)
   - **For AI systems, O(1) vs O(n) can mean 100x+ speedup**

4. **Practical Tokenization**
   - Tokenizers map words to integer IDs for neural networks
   - Dictionary-based lookup is essential for performance
   - Special tokens ([PAD], [UNK], [BOS], [EOS]) handle edge cases

### Why This Matters for AI

> **Key Insight:** Modern LLMs process billions of tokens during training. A tokenizer that's 100x slower due to poor data structure choice could mean the difference between training a model in 1 week vs. 100 weeks. Data structure selection is not an academic exercise; it's a **production engineering decision** that affects cost, time, and feasibility.

### Connection to Real AI Systems

- **GPT-4:** Uses a dictionary-based tokenizer with ~100K vocabulary
- **BERT:** Uses WordPiece tokenization with similar dictionary lookup
- **LLaMA:** Uses SentencePiece tokenization, still with hash-based lookup

All production AI systems use O(1) data structures for vocabulary lookup. There is no alternative at scale.

### Next Steps

In **Module 2: Pythonic Patterns for AI**, you'll learn:
- List and dictionary comprehensions for concise data transformation
- Generator functions for memory-efficient data streaming
- Building PyTorch-style DataLoaders from scratch

---

## Try It Yourself: Exercises

### Exercise 1: Extend the Tokenizer

**Task:** Add the following methods to the `SimpleTokenizer` class:
1. `encode_with_special_tokens(text)` - Add [BOS] at start and [EOS] at end
2. `get_vocab_size()` - Return total vocabulary size
3. `get_token_frequency(word)` - Return how often a word appears in the corpus

```python
# Your code here
class ExtendedTokenizer(SimpleTokenizer):
    def encode_with_special_tokens(self, text):
        # Add your implementation
        pass
```

### Exercise 2: Character-Level Tokenizer

**Task:** Build a character-level tokenizer (instead of word-level) that:
1. Creates a vocabulary of unique characters from the corpus
2. Assigns each character a unique ID
3. Implements `encode()` and `decode()` methods

**Hint:** Character-level tokenizers are simpler but result in longer sequences.

### Exercise 3: Performance Comparison

**Task:** Create your own performance experiment:
1. Generate a vocabulary of 10,000 words
2. Create both list and dictionary representations
3. Time 1,000 random lookups for each
4. Plot the results

### Exercise 4: Nested Dictionary Challenge

**Task:** Create a tokenizer configuration dictionary that includes:
```python
config = {
    "metadata": {
        "name": "my_tokenizer",
        "version": "1.0",
        "special_tokens": {...}
    },
    "vocab": {...},
    "statistics": {
        "most_frequent": [...],
        "coverage": 0.95
    }
}
```

Then write a function to safely extract values using `.get()` with appropriate defaults.

### Exercise 5: Real-World Application

**Task:** Download a real text dataset (e.g., a book from Project Gutenberg) and:
1. Build a tokenizer from it
2. Analyze vocabulary size vs. corpus size
3. Compare performance with your small example

---

**Solutions will be provided in the next session.**

**Need help?** Review the code examples above and experiment with modifications.